# Assignment 26: Groq API Chatbot, RAG & FastAPI Serving
**Student:** Parth Dadhaniya  
**Course:** Generative AI Engineering  

This notebook covers:
- Setting up the Groq Python SDK and invoking low-latency LLM inference
- Building core chatbot logic with system and user messages
- Constructing a grounded RAG pipeline with Chroma vector store and structured prompts
- Building and serving REST APIs using FastAPI with Pydantic validation
- Testing endpoints and analyzing observations on Groq LPUs vs GPUs

## 1. Environment & Groq Configuration
Initialize settings and check for `GROQ_API_KEY`.

In [ ]:
import os
import time
import warnings
warnings.filterwarnings('ignore')

import config
print('Configured Model :', config.DEFAULT_MODEL)
print('Groq API Key     :', 'CONFIGURED' if config.GROQ_API_KEY else 'OFFLINE / LOCAL TEST MODE')

## 2. Part 1 — Groq API Setup & Basic Chat
### Task 1: Basic Chat Prompt

In [ ]:
prompt = 'Explain in 2 sentences why Groq LPUs provide faster inference than traditional GPUs.'
messages = [{'role': 'user', 'content': prompt}]

t0 = time.time()
response = config.call_groq(messages)
print(f'Response (took {round(time.time() - t0, 3)}s):\n', response.strip())

### Task 2: Core Chatbot Function (`groq_chat`)

In [ ]:
from task2_groq_chatbot import groq_chat

for q in ['What is the capital of France?', 'Why is Groq fast compared to standard cloud LLM hosting?']:
    print('Q:', q)
    print('A:', groq_chat(q).strip(), '\n')

## 3. Part 2 — Groq + RAG Pipeline
### Tasks 3 & 4: Grounded Retrieval & Structured Prompt Template

In [ ]:
from task3_task4_groq_rag import groq_rag_pipeline

rag_q = 'What is the annual leave policy according to the employee handbook?'
res = groq_rag_pipeline(rag_q)
print('Query  :', res['query'])
print('Answer :', res['answer'])
print('Sources:', res['sources'])
print('Latency:', res['latency_seconds'], 's')

## 4. Part 3 & 4 — FastAPI Application & Testing
### Tasks 5, 6, 7 & 9: Test FastAPI Endpoints using TestClient

In [ ]:
from fastapi.testclient import TestClient
from app import app

client = TestClient(app)

# 1. Test /health
h_res = client.get('/health')
print('GET /health:', h_res.status_code, h_res.json())

# 2. Test POST /chat (Direct)
c_res = client.post('/chat', json={'query': 'Why is Groq fast?', 'use_rag': False})
print('\nPOST /chat (Direct):', c_res.status_code)
print('Answer:', c_res.json()['answer'])

# 3. Test POST /chat (RAG)
r_res = client.post('/chat', json={'query': 'What is the employee leave policy?', 'use_rag': True})
print('\nPOST /chat (RAG):', r_res.status_code)
print('Answer :', r_res.json()['answer'])
print('Sources:', r_res.json()['sources'])

## 5. Part 5 — Observations & Insights
### Task 10: Observations & Insights

**1. Why is Groq suitable for real-time applications?**  
Groq LPUs eliminate memory bandwidth bottlenecks with on-chip SRAM and deterministic compiler scheduling, generating 300-500+ tokens per second. This enables near-instant response times critical for real-time voice and chat applications.

**2. Groq vs OpenAI latency comparison (conceptual):**  
- **Groq:** Focused on raw token generation speed for open weights (Llama 3, Mistral) with minimal latency and high throughput.
- **OpenAI:** High multimodal and reasoning depth (GPT-4o), but incurs higher queuing and round-trip latency.

**3. Benefits of an API-first GenAI architecture:**  
- Decoupled frontend clients (web, mobile, Slack).
- Centralized secrets management and rate limiting.
- Ability to swap underlying LLM backends (Groq, OpenAI, Ollama) without modifying frontend code.